# Agentic Landscape Figures

This notebook regenerates every SVG figure used by `agentic-report-draft.md`.

Workflow:

1. Run the setup cell first.
2. Run or edit one figure cell at a time.
3. Each figure writes to `reports/260601-agentic_landscape/figures/`.


In [ ]:
from pathlib import Path
import ast
import html
import math
from collections import Counter, defaultdict

import pandas as pd

BASE = Path.cwd()
for candidate in [BASE, *BASE.parents]:
    if (candidate / "data" / "agentic-ai-projects.csv").exists():
        BASE = candidate
        break
else:
    raise FileNotFoundError("Could not find data/agentic-ai-projects.csv from current working directory")

DATA_FILE = BASE / "data" / "agentic-ai-projects.csv"
OUT_DIR = BASE / "reports" / "260601-agentic_landscape"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = [
    "2025-05", "2025-06", "2025-07", "2025-08", "2025-09", "2025-10",
    "2025-11", "2025-12", "2026-01", "2026-02", "2026-03", "2026-04",
]

COLORS = {
    "blue": "#2563eb",
    "cyan": "#0891b2",
    "green": "#16a34a",
    "orange": "#ea580c",
    "purple": "#7c3aed",
    "pink": "#db2777",
    "gray": "#64748b",
    "dark": "#0f172a",
    "grid": "#e2e8f0",
    "muted": "#475569",
    "bg": "#ffffff",
}


def parse_trend(value):
    try:
        values = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return [None if v is None or (isinstance(v, float) and math.isnan(v)) else float(v) for v in values]


def escape(value):
    return html.escape(str(value), quote=True)


def svg_text(x, y, text, size=12, weight=400, fill=None, anchor="start"):
    fill = fill or COLORS["dark"]
    return (
        f'<text x="{x}" y="{y}" font-family="Inter, Arial, sans-serif" '
        f'font-size="{size}" font-weight="{weight}" fill="{fill}" '
        f'text-anchor="{anchor}">{escape(text)}</text>'
    )


def save_svg(name, width, height, body):
    content = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" role="img">'
        f'<rect width="{width}" height="{height}" fill="{COLORS["bg"]}"/>'
        f'{body}</svg>\n'
    )
    path = FIG_DIR / name
    path.write_text(content, encoding="utf-8")
    print(f"Wrote {path.relative_to(BASE)}")


def category_counts(df):
    counts = Counter()
    openrank = defaultdict(float)
    stars = defaultdict(int)
    for _, row in df.iterrows():
        cats = [c.strip() for c in str(row["categories"]).split("|") if c.strip()]
        score = row["openrank_current"]
        score = 0 if pd.isna(score) else float(score)
        for cat in cats:
            counts[cat] += 1
            openrank[cat] += score
            stars[cat] += int(row["stars"] or 0)
    rows = [
        {"category": cat, "projects": count, "openrank": round(openrank[cat], 2), "stars": stars[cat]}
        for cat, count in counts.items()
    ]
    return sorted(rows, key=lambda x: (x["projects"], x["openrank"]), reverse=True)


def trend_change(row):
    values = [v for v in row["trend"] if v is not None]
    if len(values) < 2:
        return None
    start = values[0]
    end = values[-1]
    pct = None if start == 0 else (end - start) / start * 100
    return {"start": start, "end": end, "absolute": end - start, "pct": pct}


def draw_bar_chart(rows, name, title, label_key, value_key, width=1100, height=650):
    left = 275
    right = 80
    top = 82
    row_h = 36
    chart_w = width - left - right
    max_value = max(r[value_key] for r in rows) or 1
    parts = [svg_text(32, 42, title, 24, 700)]
    for i, row in enumerate(rows):
        y = top + i * row_h
        value = row[value_key]
        bar_w = chart_w * value / max_value
        parts.append(svg_text(left - 14, y + 20, row[label_key], 13, 500, COLORS["dark"], "end"))
        parts.append(f'<rect x="{left}" y="{y}" width="{bar_w:.1f}" height="22" rx="4" fill="{COLORS["blue"]}"/>')
        parts.append(svg_text(left + bar_w + 8, y + 17, f"{value:,.0f}", 12, 600, COLORS["muted"]))
    save_svg(name, width, height, "".join(parts))


def draw_line_chart(df, repos, name, title, width=1100, height=650):
    left = 78
    right = 190
    top = 78
    bottom = 78
    chart_w = width - left - right
    chart_h = height - top - bottom
    colors = [COLORS["blue"], COLORS["orange"], COLORS["green"], COLORS["purple"], COLORS["pink"], COLORS["cyan"]]
    max_y = 1
    for repo in repos:
        trend = df.loc[df["repo_name"] == repo, "trend"].iloc[0]
        max_y = max(max_y, max([v for v in trend if v is not None] or [0]))
    max_y = math.ceil(max_y / 100) * 100 if max_y > 100 else math.ceil(max_y / 10) * 10
    parts = [svg_text(32, 42, title, 24, 700)]
    for i in range(5):
        y = top + chart_h * i / 4
        value = max_y * (4 - i) / 4
        parts.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + chart_w}" y2="{y:.1f}" stroke="{COLORS["grid"]}"/>')
        parts.append(svg_text(left - 12, y + 4, f"{value:.0f}", 11, 400, COLORS["muted"], "end"))
    for i, month in enumerate(MONTHS):
        if i % 2 == 0 or i == len(MONTHS) - 1:
            x = left + chart_w * i / (len(MONTHS) - 1)
            parts.append(svg_text(x, height - 34, month[2:], 11, 400, COLORS["muted"], "middle"))
    for idx, repo in enumerate(repos):
        trend = df.loc[df["repo_name"] == repo, "trend"].iloc[0]
        pts = []
        for i, value in enumerate(trend):
            if value is None:
                continue
            x = left + chart_w * i / (len(MONTHS) - 1)
            y = top + chart_h * (1 - value / max_y)
            pts.append((x, y))
        if len(pts) < 2:
            continue
        color = colors[idx % len(colors)]
        path = " ".join([f"{x:.1f},{y:.1f}" for x, y in pts])
        parts.append(f'<polyline points="{path}" fill="none" stroke="{color}" stroke-width="3"/>')
        for x, y in pts[-1:]:
            parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="4" fill="{color}"/>')
        legend_y = top + 26 + idx * 26
        parts.append(f'<rect x="{width - right + 26}" y="{legend_y - 10}" width="14" height="14" rx="3" fill="{color}"/>')
        parts.append(svg_text(width - right + 48, legend_y + 2, repo, 12, 600, COLORS["dark"]))
    save_svg(name, width, height, "".join(parts))


def draw_scatter(df, name, title, width=1100, height=700):
    left = 92
    right = 72
    top = 88
    bottom = 78
    chart_w = width - left - right
    chart_h = height - top - bottom
    rows = df[df["openrank_current"] > 0].copy()
    rows["log_stars"] = rows["stars"].apply(lambda v: math.log10(max(v, 1)))
    rows["log_openrank"] = rows["openrank_current"].apply(lambda v: math.log10(max(v, 0.01)))
    min_x, max_x = rows["log_stars"].min(), rows["log_stars"].max()
    min_y, max_y = rows["log_openrank"].min(), rows["log_openrank"].max()
    parts = [svg_text(32, 42, title, 24, 700)]
    for i in range(5):
        x = left + chart_w * i / 4
        y = top + chart_h * i / 4
        parts.append(f'<line x1="{x:.1f}" y1="{top}" x2="{x:.1f}" y2="{top + chart_h}" stroke="{COLORS["grid"]}"/>')
        parts.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + chart_w}" y2="{y:.1f}" stroke="{COLORS["grid"]}"/>')
    for _, row in rows.iterrows():
        x = left + chart_w * (row["log_stars"] - min_x) / (max_x - min_x)
        y = top + chart_h * (1 - (row["log_openrank"] - min_y) / (max_y - min_y))
        r = 3.5 + min(9, math.sqrt(row["stars"]) / 95)
        color = COLORS["blue"] if "Coding Agent" in row["categories"] else COLORS["cyan"]
        parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" fill="{color}" opacity="0.58"/>')
    labels = ["openclaw/openclaw", "anthropics/claude-code", "anomalyco/opencode", "openai/codex", "google-gemini/gemini-cli"]
    for repo in labels:
        row = rows[rows["repo_name"] == repo]
        if row.empty:
            continue
        row = row.iloc[0]
        x = left + chart_w * (row["log_stars"] - min_x) / (max_x - min_x)
        y = top + chart_h * (1 - (row["log_openrank"] - min_y) / (max_y - min_y))
        parts.append(svg_text(x + 10, y - 8, repo, 12, 700, COLORS["dark"]))
    parts.append(svg_text(left + chart_w / 2, height - 28, "Stars, log scale", 13, 600, COLORS["muted"], "middle"))
    parts.append(svg_text(26, top + chart_h / 2, "OpenRank, log scale", 13, 600, COLORS["muted"], "middle"))
    save_svg(name, width, height, "".join(parts))


def load_agentic_data():
    df = pd.read_csv(DATA_FILE)
    openrank_columns = sorted([col for col in df.columns if col.startswith("openrank_") and col != "openrank_trend"])
    openrank_col = openrank_columns[-1] if openrank_columns else "openrank_latest"
    df["openrank_current"] = pd.to_numeric(df[openrank_col], errors="coerce")
    df["trend"] = df["openrank_trend"].fillna("[]").apply(parse_trend)
    df["change"] = df.apply(trend_change, axis=1)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    return df, openrank_col


df, openrank_col = load_agentic_data()
print(f"Loaded {len(df)} projects from {DATA_FILE.relative_to(BASE)}; OpenRank field: {openrank_col}")


## 1. Category Distribution

In [ ]:
cats = category_counts(df)
draw_bar_chart(
    cats[:15],
    "category-distribution.svg",
    "Agentic AI Categories by Project Count",
    "category",
    "projects",
)


## 2. Top OpenRank Projects

In [ ]:
ranked_df = df[df["openrank_current"].notna()].copy()
top_openrank = ranked_df.sort_values("openrank_current", ascending=False).head(15)

draw_bar_chart(
    top_openrank[["repo_name", "openrank_current"]]
    .rename(columns={"openrank_current": "openrank"})
    .to_dict("records"),
    "top-openrank-projects.svg",
    "Top Projects by Latest OpenRank",
    "repo_name",
    "openrank",
)


## 3. Language Mix

In [ ]:
lang_counts = df["language"].fillna("Unknown").replace("", "Unknown").value_counts().head(10)

draw_bar_chart(
    [{"language": k, "projects": int(v)} for k, v in lang_counts.items()],
    "language-mix.svg",
    "Language Mix by Project Count",
    "language",
    "projects",
    height=520,
)


## 4. OpenRank Trends for Leading Projects

In [ ]:
leading_repos = [
    "openclaw/openclaw",
    "anthropics/claude-code",
    "anomalyco/opencode",
    "openai/codex",
    "google-gemini/gemini-cli",
    "n8n-io/n8n",
]

draw_line_chart(
    df,
    leading_repos,
    "openrank-trends-leading-projects.svg",
    "OpenRank Trends for Leading Agentic Projects",
)


## 5. Stars vs OpenRank

In [ ]:
draw_scatter(
    df,
    "stars-vs-openrank.svg",
    "Stars vs OpenRank: Attention and Community Activity",
)


## 6. GitHub Jan-Apr Active Metrics

In [ ]:
github_activity = [
    {"year": 2017, "repos": 7148267, "developers": 3499277, "bots": 112},
    {"year": 2018, "repos": 8805571, "developers": 4497820, "bots": 575},
    {"year": 2019, "repos": 10376397, "developers": 5265963, "bots": 1136},
    {"year": 2020, "repos": 14117929, "developers": 6292574, "bots": 1367},
    {"year": 2021, "repos": 18584580, "developers": 7419376, "bots": 1745},
    {"year": 2022, "repos": 21229477, "developers": 8454952, "bots": 2428},
    {"year": 2023, "repos": 24959936, "developers": 9670747, "bots": 3465},
    {"year": 2024, "repos": 23626210, "developers": 10852722, "bots": 5372},
    {"year": 2025, "repos": 26318045, "developers": 12328829, "bots": 7871},
    {"year": 2026, "repos": 27106771, "developers": 13016143, "bots": 17285},
]


def draw_github_activity(rows, name="github-jan-apr-active-metrics.svg", width=920, height=520):
    left, right, top, bottom = 82, 36, 48, 86
    x0, x1 = left, width - right
    y0, y1 = top, height - bottom
    chart_w = x1 - x0
    chart_h = y1 - y0
    left_max = max(max(r["repos"] for r in rows), max(r["developers"] for r in rows))
    right_max = max(r["bots"] for r in rows)

    def x_at(i):
        return x0 + chart_w * i / (len(rows) - 1)

    def y_left(value):
        return y1 - chart_h * value / left_max

    def y_right(value):
        return y1 - chart_h * value / right_max

    def fmt_m(value):
        return f"{round(value / 1_000_000):.0f}M"

    def fmt_k(value):
        return f"{round(value / 1000):.0f}k"

    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        '<style>text{font-family:Arial,Helvetica,sans-serif;fill:#111827}.title{font-size:22px;font-weight:700}.label{font-size:12px;fill:#4b5563}.legend{font-size:13px}.tick{font-size:11px;fill:#6b7280}</style>',
        '<text x="82" y="30" class="title">GitHub Activity, Jan-Apr by Year</text>',
        '<text x="82" y="50" class="label">Distinct repositories, developers, and bot/app actors observed in OpenDigger GitHub event stream</text>',
    ]

    for i in range(5):
        y = y0 + chart_h * i / 4
        left_value = left_max * (4 - i) / 4
        right_value = right_max * (4 - i) / 4
        parts.append(f'<line x1="{x0}" y1="{y:.1f}" x2="{x1}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        parts.append(f'<text x="74" y="{y + 4:.1f}" text-anchor="end" class="tick">{fmt_m(left_value)}</text>')
        parts.append(f'<text x="892" y="{y + 4:.1f}" class="tick">{fmt_k(right_value)}</text>')

    parts.extend([
        f'<line x1="{x0}" y1="{y0}" x2="{x0}" y2="{y1}" stroke="#9ca3af"/>',
        f'<line x1="{x1}" y1="{y0}" x2="{x1}" y2="{y1}" stroke="#9ca3af"/>',
        f'<line x1="{x0}" y1="{y1}" x2="{x1}" y2="{y1}" stroke="#9ca3af"/>',
    ])

    for i, row in enumerate(rows):
        parts.append(f'<text x="{x_at(i):.1f}" y="458" text-anchor="middle" class="tick">{row["year"]}</text>')

    series = [
        ("repos", "#2563eb", y_left),
        ("developers", "#16a34a", y_left),
        ("bots", "#dc2626", y_right),
    ]
    for key, color, y_fn in series:
        points = " ".join(f'{x_at(i):.1f},{y_fn(row[key]):.1f}' for i, row in enumerate(rows))
        parts.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="3"/>')
        for i, row in enumerate(rows):
            parts.append(f'<circle cx="{x_at(i):.1f}" cy="{y_fn(row[key]):.1f}" r="4" fill="{color}"/>')

    parts.extend([
        '<circle cx="82" cy="486" r="5" fill="#2563eb"/>',
        '<text x="92" y="490" class="legend">Active repos</text>',
        '<circle cx="292" cy="486" r="5" fill="#16a34a"/>',
        '<text x="302" y="490" class="legend">Active developers</text>',
        '<circle cx="502" cy="486" r="5" fill="#dc2626"/>',
        '<text x="512" y="490" class="legend">Active bot/app actors</text>',
        '<text x="82" y="510" class="label">Left axis: repos/developers. Right axis: bot/app actors. Source: OpenDigger ClickHouse opensource.events.</text>',
    ])

    content = f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">' + "".join(parts) + "</svg>\n"
    path = FIG_DIR / name
    path.write_text(content, encoding="utf-8")
    print(f"Wrote {path.relative_to(BASE)}")


draw_github_activity(github_activity)


## 7. Hugging Face Derivatives by Organization


In [ ]:
hf_derivatives_by_org = [
    {"organization": "Alibaba (incl Qwen)", "derivatives_k": 115.0},
    {"organization": "Google", "derivatives_k": 72.0},
    {"organization": "Meta (incl Facebook)", "derivatives_k": 46.0},
    {"organization": "Black Forest Labs", "derivatives_k": 38.0},
    {"organization": "DistilBERT", "derivatives_k": 14.0},
    {"organization": "Microsoft", "derivatives_k": 14.0},
    {"organization": "Mistral AI", "derivatives_k": 12.0},
    {"organization": "Stability AI", "derivatives_k": 11.0},
    {"organization": "OpenAI", "derivatives_k": 11.0},
    {"organization": "DeepSeek", "derivatives_k": 6.5},
]

def draw_hf_derivatives_chart(rows):
    width, height = 760, 430
    left, top = 195, 46
    chart_w, row_h, bar_h = 548, 34, 31
    max_value = 120.0
    parts = [
        '<rect width="760" height="430" fill="#ffffff"/>',
        svg_text(width / 2, 28, "Derivatives on Hugging Face by Organization", 25, 800, "#050505", "middle"),
        f'<g transform="translate({left},{top})">',
    ]
    for tick in [0, 20, 40, 60, 80, 100, 120]:
        x = chart_w * tick / max_value
        parts.append(f'<line x1="{x:.0f}" y1="0" x2="{x:.0f}" y2="340" stroke="#e9e9e9" stroke-width="1"/>')
        parts.append(svg_text(x, 368, "0" if tick == 0 else f"{tick * 1000:.0f}", 15, 800, "#050505", "middle"))
    for i, row in enumerate(rows):
        y = i * row_h
        label_y = y + 20
        bar_w = chart_w * row["derivatives_k"] / max_value
        value = f'{row["derivatives_k"]:.0f}k' if row["derivatives_k"] >= 10 else f'{row["derivatives_k"]:.1f}k'
        parts.append(svg_text(-10, label_y, row["organization"], 15, 800, "#050505", "end"))
        parts.append(f'<rect x="0" y="{y}" width="{bar_w:.1f}" height="{bar_h}" fill="#0b57b7"/>')
        parts.append(svg_text(max(bar_w - 9, 28), label_y, value, 13, 400, "#d8e7ff", "end"))
    parts.append('</g>')
    save_svg("hf-derivatives-by-organization.svg", width, height, ''.join(parts))

draw_hf_derivatives_chart(hf_derivatives_by_org)
hf_derivatives_by_org


## 8. OpenRouter Token Usage by Organization


In [ ]:
openrouter_monthly_models = [
    ("Claude Sonnet 4.6", "Anthropic", 5.84),
    ("Hy3 preview (free)", "Tencent", 5.58),
    ("DeepSeek V3.2", "DeepSeek", 4.83),
    ("Gemini 3 Flash Preview", "Google", 4.57),
    ("Kimi K2.6", "Moonshot AI", 4.51),
    ("MiniMax M2.7", "MiniMax", 3.74),
    ("Claude Opus 4.6", "Anthropic", 3.61),
    ("Claude Opus 4.7", "Anthropic", 3.08),
    ("MiniMax M2.5", "MiniMax", 2.84),
    ("Grok 4.1 Fast", "xAI", 2.78),
    ("Gemini 2.5 Flash Lite", "Google", 2.64),
    ("Nemotron 3 Super (free)", "NVIDIA", 2.58),
    ("Gemini 2.5 Flash", "Google", 2.52),
    ("Step 3.5 Flash", "StepFun", 2.21),
    ("MiMo-V2-Pro", "Xiaomi", 2.21),
    ("GPT-5.4", "OpenAI", 1.89),
    ("gpt-oss-120b", "OpenAI", 1.75),
    ("GLM 5.1", "Z.ai", 1.74),
    ("Gemini 3.1 Pro Preview", "Google", 1.65),
    ("Kimi K2.5", "Moonshot AI", 1.49),
]

openrouter_trending_models = [
    ("Ring-2.6-1T (free)", "inclusionAI", 57.0),
    ("Gemini 3.1 Flash Lite", "Google", 15.1),
    ("Hy3 preview", "Tencent", 435.0),
    ("Ling-2.6-1T", "inclusionAI", 0.656),
    ("Nemotron 3 Nano Omni", "NVIDIA", 0.106),
    ("Grok 3 Beta", "xAI", 0.000044),
    ("GPT-4o Mini Transcribe", "OpenAI", 0.0242),
    ("Qwen VL Max", "Qwen", 2.18),
    ("GPT-4o Transcribe", "OpenAI", 0.0199),
    ("Trinity Large Thinking", "Arcee AI", 96.3),
    ("Llama 3.2 1B Instruct", "Meta Llama", 1.09),
    ("all-mpnet-base-v2", "Sentence Transformers", 0.0297),
    ("Owl Alpha", "OpenRouter", 349.0),
    ("Granite 4.1 8B", "IBM Granite", 2.26),
    ("Hermes 2 Pro - Llama-3 8B", "Nous Research", 0.1),
    ("GPT-3.5 Turbo 16k", "OpenAI", 0.00741),
    ("Qwen3.6 35B A3B", "Qwen", 29.8),
    ("Reka Flash 3", "Reka AI", 0.053),
    ("Laguna M.1 (free)", "Poolside", 194.0),
    ("Mistral Embed 2312", "Mistral AI", 0.224),
]

def aggregate_models(models):
    totals = defaultdict(float)
    for _, org, tokens in models:
        totals[org] += tokens
    return [
        {"organization": org, "tokens": round(tokens, 3)}
        for org, tokens in sorted(totals.items(), key=lambda item: item[1], reverse=True)[:10]
    ]

def draw_openrouter_org_chart(rows, name, title, unit, max_value, ticks):
    width, height = 760, 430
    left, top = 195, 46
    chart_w, row_h, bar_h = 548, 34, 31
    def format_value(value):
        if unit == "T":
            return f"{value:.1f}T"
        if value >= 100:
            return f"{value:.0f}B"
        if value >= 10:
            return f"{value:.1f}B"
        if value >= 1:
            return f"{value:.2g}B"
        return f"{value * 1000:.0f}M"
    parts = [
        '<rect width="760" height="430" fill="#ffffff"/>',
        svg_text(width / 2, 28, title, 24, 800, "#050505", "middle"),
        f'<g transform="translate({left},{top})">',
    ]
    for tick in ticks:
        x = chart_w * tick / max_value
        tick_label = "0" if tick == 0 else f"{tick:g}{unit}"
        parts.append(f'<line x1="{x:.0f}" y1="0" x2="{x:.0f}" y2="340" stroke="#e9e9e9" stroke-width="1"/>')
        parts.append(svg_text(x, 368, tick_label, 15, 800, "#050505", "middle"))
    for i, row in enumerate(rows):
        y = i * row_h
        label_y = y + 20
        bar_w = chart_w * row["tokens"] / max_value
        parts.append(svg_text(-10, label_y, row["organization"], 15, 800, "#050505", "end"))
        parts.append(f'<rect x="0" y="{y}" width="{bar_w:.1f}" height="{bar_h}" fill="#0b57b7"/>')
        value_x = bar_w - 9 if bar_w >= 50 else bar_w + 8
        anchor = "end" if bar_w >= 50 else "start"
        fill = "#d8e7ff" if bar_w >= 50 else "#0b57b7"
        parts.append(svg_text(value_x, label_y, format_value(row["tokens"]), 13, 400, fill, anchor))
    parts.append('</g>')
    save_svg(name, width, height, ''.join(parts))

openrouter_monthly_org_rows = aggregate_models(openrouter_monthly_models)
openrouter_trending_org_rows = aggregate_models(openrouter_trending_models)

draw_openrouter_org_chart(
    openrouter_monthly_org_rows,
    "openrouter-token-usage-by-organization-monthly.svg",
    "Monthly Token Usages on OpenRouter by Organization",
    "T",
    12.53,
    [0, 3, 6, 9, 12],
)
draw_openrouter_org_chart(
    openrouter_trending_org_rows,
    "openrouter-token-usage-by-organization.svg",
    "Trending Token Usages on OpenRouter by Organization",
    "B",
    450,
    [0, 100, 200, 300, 400],
)
{"monthly": openrouter_monthly_org_rows, "trending": openrouter_trending_org_rows}


## 9. Hugging Face Cumulative Model Downloads by Organization


In [ ]:
hf_alltime_downloads_by_author = [
    {"organization": "sentence-transformers", "downloads": 5_489_134_140},
    {"organization": "google-bert", "downloads": 3_872_452_629},
    {"organization": "OpenAI", "downloads": 2_748_821_810},
    {"organization": "FacebookAI", "downloads": 2_600_143_054},
    {"organization": "facebook", "downloads": 2_422_713_682},
    {"organization": "MIT", "downloads": 2_186_668_292},
    {"organization": "Google", "downloads": 2_095_013_844},
    {"organization": "timm", "downloads": 2_091_030_968},
    {"organization": "Microsoft", "downloads": 2_087_312_616},
    {"organization": "Qwen", "downloads": 2_067_397_564},
]

def draw_hf_alltime_downloads_chart(rows):
    width, height = 760, 430
    left, top = 195, 46
    chart_w, row_h, bar_h = 548, 34, 31
    max_value = 5_500_000_000
    parts = [
        '<rect width="760" height="430" fill="#ffffff"/>',
        svg_text(width / 2, 28, "Cumulative Model Downloads on Hugging Face by Organization", 23, 800, "#050505", "middle"),
        f'<g transform="translate({left},{top})">',
    ]
    for tick in [0, 1_000_000_000, 2_000_000_000, 3_000_000_000, 4_000_000_000, 5_000_000_000]:
        x = chart_w * tick / max_value
        label = "0" if tick == 0 else f"{tick // 1_000_000_000:.0f}B"
        parts.append(f'<line x1="{x:.0f}" y1="0" x2="{x:.0f}" y2="340" stroke="#e9e9e9" stroke-width="1"/>')
        parts.append(svg_text(x, 368, label, 15, 800, "#050505", "middle"))
    for i, row in enumerate(rows):
        y = i * row_h
        label_y = y + 20
        bar_w = chart_w * row["downloads"] / max_value
        value = f'{row["downloads"] / 1_000_000_000:.1f}B'
        parts.append(svg_text(-10, label_y, row["organization"], 15, 800, "#050505", "end"))
        parts.append(f'<rect x="0" y="{y}" width="{bar_w:.1f}" height="{bar_h}" fill="#0b57b7"/>')
        parts.append(svg_text(max(bar_w - 9, 18), label_y, value, 13, 400, "#d8e7ff", "end"))
    parts.append('</g>')
    save_svg("hf-model-downloads-alltime-by-organization.svg", width, height, ''.join(parts))

draw_hf_alltime_downloads_chart(hf_alltime_downloads_by_author)
hf_alltime_downloads_by_author
